In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

# Work on train split only
train_data = dataset["train"].shuffle(seed=42).select(range(5000))
test_data = dataset["test"].shuffle(seed=42).select(range(1000))

# Create validation split from train
train_test = train_data.train_test_split(test_size=0.1)

train_ds = train_test["train"]
val_ds = train_test["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

# For causal language modeling, input_ids are often used as labels
train_ds = train_ds.map(lambda examples: {'labels': examples['input_ids']})
val_ds = val_ds.map(lambda examples: {'labels': examples['input_ids']})

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
import logging
from transformers import GPT2LMHeadModel, Trainer, TrainingArguments

# Suppress specific warnings from the transformers library regarding unexpected keys
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

model = GPT2LMHeadModel.from_pretrained("gpt2")

training_args = TrainingArguments(
    output_dir="./baseline",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=500,
    logging_steps=100,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

trainer.train()

The following layers were not sharded: transformer.h.*.attn.c_attn.weight, transformer.h.*.ln_1.weight, transformer.h.*.mlp.c_proj.bias, transformer.h.*.attn.c_attn.bias, transformer.h.*.ln_2.bias, transformer.wte.weight, transformer.h.*.mlp.c_fc.weight, transformer.h.*.attn.c_proj.bias, lm_head.weight, transformer.h.*.ln_1.bias, transformer.ln_f.weight, transformer.ln_f.bias, transformer.h.*.ln_2.weight, transformer.wpe.weight, transformer.h.*.mlp.c_proj.weight, transformer.h.*.attn.c_proj.weight, transformer.h.*.mlp.c_fc.bias


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import BitsAndBytesConfig

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16"
)

model_q = GPT2LMHeadModel.from_pretrained(
    "gpt2",
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

peft_model = get_peft_model(model_q, lora_config)
peft_model.print_trainable_parameters()

In [ ]:
trainer_q = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)

trainer_q.train()

In [ ]:
import math

baseline_eval = trainer.evaluate()
qlora_eval = trainer_q.evaluate()

baseline_perplexity = math.exp(baseline_eval["eval_loss"])
qlora_perplexity = math.exp(qlora_eval["eval_loss"])

print("Baseline Loss:", baseline_eval["eval_loss"])
print("Baseline Perplexity:", baseline_perplexity)

print("QLoRA Loss:", qlora_eval["eval_loss"])
print("QLoRA Perplexity:", qlora_perplexity)

In [ ]:
def generate_text(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "The movie was"

print("Baseline Output:\n", generate_text(model, prompt))
print("\nQLoRA Output:\n", generate_text(peft_model, prompt))

In [ ]:
import matplotlib.pyplot as plt

baseline_logs = trainer.state.log_history
qlora_logs = trainer_q.state.log_history

def extract_loss(logs):
    return [x["loss"] for x in logs if "loss" in x]

plt.plot(extract_loss(baseline_logs), label="Baseline")
plt.plot(extract_loss(qlora_logs), label="QLoRA")

plt.legend()
plt.title("Training Loss Comparison")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.show()